In [ ]:
import os
import sys
import subprocess

# --- CONFIGURAZIONE DEL PROGETTO ---
GIT_REPO_URL = "https://github.com/Biobay/DeepLearning"
BRANCH_NAME = "ESPERIMENTO1"
PROJECT_DIR = "DeepLearning"

# =============================================================================
# 1. SETUP DELL'AMBIENTE (VERSIONE AGGRESSIVA E SICURA)
# =============================================================================

# Clona il repository se non esiste
if not os.path.exists(PROJECT_DIR):
    print(f"Clonazione del branch '{BRANCH_NAME}' da '{GIT_REPO_URL}'...")
    try:
        subprocess.run(['git', 'clone', '--branch', BRANCH_NAME, GIT_REPO_URL, PROJECT_DIR], check=True)
    except subprocess.CalledProcessError as e:
        print(f"ERRORE: Impossibile clonare il repository. Controlla l'URL e il nome del branch. Dettagli: {e}")
        sys.exit(1)
else:
    print(f"La cartella '{PROJECT_DIR}' esiste già. Salto la clonazione.")

# Naviga nella cartella del progetto
os.chdir(PROJECT_DIR)
print(f"Cartella di lavoro corrente: {os.getcwd()}")

# Installa le dipendenze con pulizia della cache e reinstallazione forzata
print("\nInstallazione delle dipendenze con pulizia della cache e reinstallazione forzata...")
print("Questo potrebbe richiedere alcuni minuti...")

pip_install_command = [
    sys.executable, '-m', 'pip', 'install',
    '--no-cache-dir',      # Non usare pacchetti dalla cache
    '--force-reinstall',   # Forza la reinstallazione anche se i pacchetti esistono già
    '-r', 'requirements.txt',
    '--quiet'              # Riduci l'output a console
]

try:
    subprocess.run(pip_install_command, check=True)
    print("Dipendenze installate con successo.")
except subprocess.CalledProcessError as e:
    print(f"ERRORE CRITICO durante l'installazione delle dipendenze: {e}")
    print("Controlla il file 'requirements.txt' e la tua connessione a internet.")
    sys.exit(1)

# Aggiorna il path di Python per permettere gli import locali
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print("Path di sistema aggiornato per gli import.")

# =============================================================================
# 2. VERIFICA DELL'AMBIENTE
# =============================================================================

# Aggiungiamo un controllo per verificare la versione di PyTorch installata
try:
    import torch
    print("-" * 60)
    print(f"VERIFICA AMBIENTE:")
    print(f"  -> Versione PyTorch installata: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"  -> GPU Trovata: {torch.cuda.get_device_name(0)}")
        print(f"  -> CUDA Disponibile: Sì")
    else:
        print(f"  -> CUDA Disponibile: No (il training verrà eseguito su CPU)")
    print("-" * 60)
except ImportError:
    print("ERRORE CRITICO: PyTorch non sembra essere stato installato correttamente.")
    sys.exit(1)

# =============================================================================
# 3. AVVIO DELL'ADDESTRAMENTO
# =============================================================================

print("\n--- Avvio del processo di addestramento ---")
try:
    # Importa i moduli del progetto DOPO l'installazione e l'aggiornamento del path
    from scripts.train import train
    import src.config as config
    print("Moduli del progetto importati con successo!")
    
    # Esegui la funzione di training
    # Nota: questa versione non cattura la history per l'analisi.
    # Se vuoi l'analisi, dovrai ripristinare la logica precedente.
    train(config)
    
    print("\n--- Processo di addestramento terminato con successo ---")

except ImportError as e:
    print(f"ERRORE: Impossibile importare i moduli del progetto. Controlla i path e la struttura delle cartelle.")
    print(f"Dettagli: {e}")
except Exception as e:
    print(f"Si è verificato un errore imprevisto durante l'addestramento: {e}")